<div style="background-color: #013220; color: white; padding: 10px;">
    <h1>Générer Excel Shap</h1>
</div>

In [ ]:
import os 
import pandas as pd
import numpy as np
from xbbg import blp
from datetime import datetime, timedelta
from Codes.ML_SUIVI import *
from Config import config_EU, config_US, config_OTHER
from Codes.BacktestEngine import merge_ticker_secondaire
%load_ext autoreload
%autoreload 2

list_noire_path = r"\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_BASE\_ ESG DATA\Liste_Noire_Exclusion.xlsx"
path_ptf_world = r"Portfolio_BT\PTF_WORLD_semi.pkl"
screen_path = r"\\groupe-ufg.com\commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\0_SCREEN_AGG\screen_aggregate.pkl"
returns_path = r"\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\0_RETURNS\returns.pkl",
path_ciq = r"\\groupe-ufg.com\commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\0_SCREEN_AGG\last_screenCIQ.pkl"
mail_output_path =  r"output_mail\\"

shap_grouped = combine_score_shap(config_EU, config_US, config_OTHER)

# Ajuster les outliers
shap_grouped = process_outlier_rows_only(shap_grouped, columns=['Dividend Avg Percentile', 'Value Avg Percentile',
        'Quality Avg Percentile', 'Mom Avg Percentile', 'LowVol Avg Percentile',
        'Growth Avg Percentile', 'Value Avg Percentile_change_1M',
        'Quality Avg Percentile_change_1M', 'Growth Avg Percentile_change_1M',
        'Value Avg Percentile_change_3M', 'Quality Avg Percentile_change_3M',
        'Growth Avg Percentile_change_3M', 'Value Avg Percentile_change_6M',
        'Quality Avg Percentile_change_6M', 'Growth Avg Percentile_change_6M',
        'Value Avg Percentile_change_12M', 'Quality Avg Percentile_change_12M',
        'Growth Avg Percentile_change_12M', 'Sector 1', 'Sector 2', 'Sector 3',
        'Sector 4', 'Sector 5', 'Sector 6', 'Sector 7', 'Sector 8', 'Sector 9',
        'Sector 10', 'Sector 11', 'Sector 12', 'Sector 13', 'Sector 14',
        'Sector 15', 'Sector 16', 'Sector 17', 'Sector 18', 'Sector 19'], sum_col="average_prediction")

last_date = shap_grouped['Date'].max()
df_screen = add_reco_analyst_multi_facteur(screen_path, path_ciq, last_date)

df_screen = merge_ticker_secondaire(df_screen)

df_total = df_screen.merge(shap_grouped,                                 
                        how="left",
                        left_on="ISIN",
                        right_on="ISIN"
                        )
df_total = add_raison_repechage(df_total, path_ptf_world)

# Rebalance les weights
df_total['Weight'] = df_total['Weight'].div(df_total['Weight'].sum(), axis=0)

# Aggregate les Impact Facto et Secto
df_total['Contrib Facto'] = df_total[['Dividend Avg Percentile', 'Value Avg Percentile',
                                'Quality Avg Percentile', 'Mom Avg Percentile', 'LowVol Avg Percentile',
                                'Growth Avg Percentile', 'Value Avg Percentile_change_1M',
                                'Quality Avg Percentile_change_1M', 'Growth Avg Percentile_change_1M',
                                'Value Avg Percentile_change_3M', 'Quality Avg Percentile_change_3M',
                                'Growth Avg Percentile_change_3M', 'Value Avg Percentile_change_6M',
                                'Quality Avg Percentile_change_6M', 'Growth Avg Percentile_change_6M',
                                'Value Avg Percentile_change_12M', 'Quality Avg Percentile_change_12M',
                                'Growth Avg Percentile_change_12M']].sum(axis=1)

df_total['Contrib Secto'] = df_total[['Sector 1', 'Sector 2', 'Sector 3',
                                'Sector 4', 'Sector 5', 'Sector 6', 'Sector 7', 'Sector 8', 'Sector 9',
                                'Sector 10', 'Sector 11', 'Sector 12', 'Sector 13', 'Sector 14',
                                'Sector 15', 'Sector 16', 'Sector 17', 'Sector 18', 'Sector 19']].sum(axis=1)
df_total = rename_cols_1(df_total)  # Change specific columns' name
df_total = rename_cols_2(df_total)  # Change contribution related columns' name
screen_original_data = get_factos()
screen_original_data = rename_cols_3(screen_original_data)  # Change original score columns' name

df_total = df_total.merge(screen_original_data,                             
                how="left",
                left_on="ISIN",
                right_on="ISIN"
                )

df_total = add_esg_list_noire(list_noire_path, df_total)
list_exclu_total = get_list_exclusion()
df_total = df_total.merge(list_exclu_total[['ISIN', 'Raison Exclusion']], how="left", on='ISIN')

# Cols to show in Excel
list_cols_excel = [
                'PTF', "Raison Repechage", "Raison Exclusion", 'Date','ISIN', 'Company SEDOL', 'Name', "Region", 'Country', 'Weight', 'Weight in MSCI WORLD', "Score ESG", 'Blacklisted', 
                'Sector',  'DVD Yield', 'Earnings Yield', 'ROE', 'PE', "Oper Margin", 'Carbon Intensity',
                'Reco Analyst', 'Multi Score', 'Score ML', 'Predicted Forward Return 1M',

                'Contrib Facto',
                'Value Contrib', 'Value Change_1M', 'Value Change_3M', 'Value Change_6M', 'Value Change_12M', 
                'Value Score', 'Value Score Change_1M', 'Value Score Change_3M', 'Value Score Change_6M', 'Value Score Change_12M', 

                'Quality Contrib', 'Quality Change_1M', 'Quality Change_3M', 'Quality Change_6M', 'Quality Change_12M',
                'Quality Score', 'Quality Score Change_1M', 'Quality Score Change_3M', 'Quality Score Change_6M', 'Quality Score Change_12M',

                'Growth Contrib', 'Growth Change_1M', 'Growth Change_3M', 'Growth Change_6M', 'Growth Change_12M', 
                'Growth Score', 'Growth Score Change_1M', 'Growth Score Change_3M', 'Growth Score Change_6M', 'Growth Score Change_12M', 

                'Dividend Contrib', 'Mom Contrib', 'LowVol Contrib',
                'Dividend Score', 'Mom Score', 'LowVol Score',
                
                'Contrib Secto',
                'Sector 1', 'Sector 2', 'Sector 3',
                'Sector 4', 'Sector 5', 'Sector 6', 'Sector 7', 'Sector 8', 'Sector 9',
                'Sector 10', 'Sector 11', 'Sector 12', 'Sector 13', 'Sector 14',
                'Sector 15', 'Sector 16', 'Sector 17', 'Sector 18', 'Sector 19'
                ]

basket = df_total[list_cols_excel]
basket_et_univ = basket.copy(deep=True)
basket_et_univ.to_excel(os.path.join(mail_output_path, r"basket_et_shap_test.xlsx"), index=False)

In [5]:
df_screen[df_screen['Name'].str.contains('A.P. Moller')]

,ISIN,Weight in MSCI WORLD,ICB19 Supersector,Name,Exchange Country Region,Exchange Country Name,ESG_ANALYST_SCORE,Reco Analyst,Multi Avg Percentile,PE LTM,EPS Growth FY1,ROE avg FY0,Oper Margin,DVD Yield FY0,Earns Yield FY0,CarbonIntensity_Sales
2172,DK0010244425,0.000095,Industrial Goods & Services,A.P. Moller - Maersk A/S Class A,West Europe,DENMARK,NaN,Underperform,4.973262,3.873128,-63.50153,13.19344,12.18445,1.14625,23.14265,NaN
2173,DK0010244508,0.000123,Industrial Goods & Services,A.P. Moller - Maersk A/S Class B,West Europe,DENMARK,5.56,Underperform,4.919786,3.897956,-63.24789,13.19344,12.18445,1.143196,22.43311,730.1593


<div style="background-color: #013220; color: white; padding: 10px;">
    <h1>Générer Les Matrices de Confusion</h1>
</div>

In [ ]:
import pandas as pd
screen_EU = pd.read_pickle(r"Output_files\SCORE_ML_EU.pkl")
screen_US = pd.read_pickle(r"Output_files\SCORE_ML_US.pkl")
screen_OTHER = pd.read_pickle(r"Output_files\SCORE_ML_OTHER.pkl")
screen_EU = screen_EU[screen_EU['Weight in STOXX EUROPE 600'] > 0].dropna(subset="Score ML")
screen_US = screen_US[screen_US['Weight in MSCI US'] > 0].dropna(subset="Score ML")
screen_OTHER = screen_OTHER[screen_OTHER['Weight in MSCI WORLD'] > 0].dropna(subset="Score ML")
screen = pd.concat([screen_EU, screen_US, screen_OTHER])


import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

# Define analysis parameters
N_QUANTILES = 5 # We will continue using deciles (10-quantiles)
LABELS = [f'Q{i+1}' for i in range(N_QUANTILES)]
LABELS[0] = 'Q1_Bottom'
LABELS[-1] = 'Q5_Top'
returns_choisi = "Weighted 1.0M return"


# Determine the time range
# Find the latest date in the data to define the time window
latest_date = screen['Date'].max()
nb_years = 10
five_years_ago = latest_date - timedelta(days=nb_years*365)

# Filter data for the last five years (historical) and the latest date
historical_data = screen[(screen['Date'] >= five_years_ago) & (screen['Date'] < latest_date)]
historical_data = transform_return_data(historical_data)
latest_data = screen[screen['Date'] == latest_date]

# Apply the function to historical and latest data
historical_classified = historical_data.groupby('Date').apply(lambda x: assign_quantiles(x, N_QUANTILES, LABELS, columns_to_quantile=["Score ML", "Multi Avg Percentile", 'Weighted 1.0M return'])).reset_index()
latest_classified = assign_quantiles(latest_data.copy(), N_QUANTILES, LABELS, columns_to_quantile=["Score ML", "Multi Avg Percentile"])



########################## Analyse Score MF vs Score ML ##########################
# Calculate metrics for each day in the historical period
historical_metrics = historical_classified.groupby('Date').apply(lambda x: calculate_monitoring_metrics(x, LABELS))

system_alert_hit_rate(historical_metrics, latest_classified, LABELS)

# Calculate normalized matrices for visualization
historical_matrices_normalized = [confusion_matrix(g['Multi_Quantile'], g['ML_Quantile'], labels=LABELS, normalize='all') 
                                for _, g in historical_classified.groupby('Date')]
average_cm = np.mean(np.array(historical_matrices_normalized), axis=0)
latest_cm = confusion_matrix(latest_classified['Multi_Quantile'], latest_classified['ML_Quantile'], labels=LABELS, normalize='all')


# ------------------------------------------------------------------
# Build the subplot grid (1 row × 3 columns)
# ------------------------------------------------------------------
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        f"{nb_years}-Year Average Matrix",
        "Latest Date Matrix",
        "Difference Matrix (Latest - Average)"
    ],
    shared_yaxes=True,
    vertical_spacing=0.1,
    horizontal_spacing=0.1,
)


# 10‑Year average matrix
fig.add_trace(
    heatmap_trace(
        z=average_cm,
        labels=LABELS,
        title="",
        cmap="Blues",
    ),
    row=1, col=1
)

# Latest‑date matrix
fig.add_trace(
    heatmap_trace(
        z=latest_cm,
        labels=LABELS,
        title="",
        cmap="Blues",
    ),
    row=1, col=2
)

fig.update_layout(
    title=dict(
        text=f"Model Behavior Comparison: Latest Date ({latest_date.date()}) vs. "
            f"{nb_years}-Year Average",
        x=0.5,
        xanchor="center",
        yanchor="top",
        font=dict(size=16)
    ),
    width=1100,
    height=500,
    autosize=False,
    margin=dict(l=80, r=80, t=100, b=80),
    showlegend=False,
)

fig.update_yaxes(title_text="Multi‑Factor Quantile",
                row=1, col=1)
fig.update_yaxes(title_text="", row=1, col=2)
fig.update_yaxes(title_text="", row=1, col=3)
fig.update_xaxes(title_text="Score ML Quantile")

fig.show()



########################## Analyse Returns vs Score ML ##########################
historical_classified['ML_Quantile'] = historical_classified['ML_Quantile'].astype("str", errors='ignore')
historical_classified['Return_Quantile'] = historical_classified['Return_Quantile'].astype("str", errors='ignore')
# Calculate normalized matrices for visualization
average_cm = confusion_matrix(historical_classified['Return_Quantile'], historical_classified['ML_Quantile'], labels=LABELS, normalize='all') 
average_cm = average_cm - ( 1 / (N_QUANTILES * N_QUANTILES)) # ajuster avec 



# ------------------------------------------------------------------
# Build the subplot grid (1 row × 3 columns)
# ------------------------------------------------------------------
fig = make_subplots(
    rows=1,
    cols=1,
    subplot_titles=[
        f"{nb_years}-Year Average Matrix"
    ],
    shared_yaxes=True,
    vertical_spacing=0.1,
    horizontal_spacing=0.1,
)

fig.add_trace(
    heatmap_trace(
        z=average_cm,
        labels=LABELS,
        title="",
        cmap="RdYlGn",
        zmax=0.002
    ),
    row=1, col=1
)

# ------------------------------------------------------------------
# Global figure layout
# ------------------------------------------------------------------
fig.update_layout(
    title=dict(
        text=f"ML Model Predictive Power Matrix<br>"
            f"Forward Return Quantile vs. Score ML Matrix"
            # f"{nb_years}-Year Average"
            " <br>",
        x=0.5,
        xanchor="center",
        yanchor="top",
        font=dict(size=16)
    ),
    width=600,
    height=500,
    autosize=False,
    margin=dict(l=80, r=80, t=100, b=80),
    showlegend=False,
)

fig.update_yaxes(title_text="Forward Return Quantile",
                row=1, col=1)
fig.update_yaxes(title_text="", row=1, col=2)
fig.update_yaxes(title_text="", row=1, col=3)
fig.update_xaxes(title_text="Score ML Quantile")

fig.show()

<div style="background-color: #013220; color: white; padding: 10px;">
    <h1>Générer Les Graph pour Mail</h1>
</div>

In [ ]:
perf = get_ptf_perf(r"Portfolio_BT\Perf_ESG_WORLD.pkl", "MSDEWIN Index", source="pickle")
plot_price(perf, start_date='2024-01-01', output=os.path.join(mail_output_path, r"perf_plot.png"), figsize=(12, 5))
perf_results = calculate_multiple_performances(perf)
print(perf_results)
metrics = calculate_portfolio_metrics(perf[['TOP ML']], perf[['Bench']])

basket = basket.dropna(subset="PTF")
bench_list = get_bench_cols(screen_path, "MSCI WORLD")
ratio_top = calculate_ratio_vs_bench(basket, bench_list, metrics)
print(ratio_top)
top_25_weekly, worst_25_weekly = top_worst_performers(returns_path, basket, n=25)

basket = map_19_to_11(basket)

create_sector_weight_comp_plot(basket, bench_list, os.path.join(mail_output_path, r"weight_comparison_secto.png"))
create_sector_return_comp_plot(basket, bench_list, returns_path, (-0.2, 0.2), os.path.join(mail_output_path, r"rdt_comparaison_secto.png"))

save_dataframe_as_image(perf_results.T, os.path.join(mail_output_path, r"periodic_perf.png"), color='#63a1c7')
save_dataframe_as_image(ratio_top, os.path.join(mail_output_path, r"metrics.png"), color='#63a1c7')
save_dataframe_as_image(top_25_weekly, os.path.join(mail_output_path, r"top_25_weekly.png"), col_width=20, color='#63a1c7')
save_dataframe_as_image(worst_25_weekly, os.path.join(mail_output_path, r"worst_25_weekly.png"), col_width=20, color='#63a1c7')

<div style="background-color: #013220; color: white; padding: 10px;">
    <h1>Envoi Mail</h1>
</div>

In [6]:
import win32com.client as win32
import os
import sys

def generate_and_send_mail(subject, recipients, body, attachments=None, embedded_images=None, tables=None, auto_send=False):
    """
    Generate and send an email using Outlook.

    Parameters:
    -----------
    subject : str
        The subject of the email.
    recipients : str or list
        Email address(es) of the recipient(s). Can be a string for a single recipient or a list for multiple.
    body : str
        The main body of the email in HTML format.
    attachments : list of str, optional
        List of file paths to be attached to the email.
    embedded_images : dict, optional
        Dictionary of image IDs and their file paths to be embedded in the email body.
    tables : list of pandas.DataFrame or list of str, optional
        List of pandas DataFrames or HTML strings to be included in the email body.
    auto_send : bool, optional
        If True, sends the email automatically. If False, displays the email draft. Default is False.

    Returns:
    --------
    None
    """
    try:
        outlook = win32.Dispatch('outlook.application')
        mail = outlook.CreateItem(0)

        # Set recipients
        if isinstance(recipients, list):
            mail.To = '; '.join(recipients)
        else:
            mail.To = recipients

        mail.Subject = subject

        # Attach files
        if attachments:
            for file_path in attachments:
                if os.path.exists(file_path):
                    try:
                        mail.Attachments.Add(file_path)
                    except Exception as e:
                        print(f"Error attaching file {file_path}: {str(e)}")
                else:
                    print(f"Warning: Attachment not found - {file_path}")

        # Embed images
        if embedded_images:
            for img_id, img_path in embedded_images.items():
                if os.path.exists(img_path):
                    try:
                        attachment = mail.Attachments.Add(img_path)
                        attachment.PropertyAccessor.SetProperty("http://schemas.microsoft.com/mapi/proptag/0x3712001F", img_id)
                    except Exception as e:
                        print(f"Error embedding image {img_path}: {str(e)}")
                else:
                    print(f"Warning: Embedded image not found - {img_path}")

        # Process tables
        html_tables = []
        if tables:
            for i, table in enumerate(tables):
                if hasattr(table, 'to_html'):  # If it's a pandas DataFrame
                    html_table = table.to_html(index=True, border=0)
                    html_table = html_table.replace(
                        '<table', f'<table style="width:100%; text-align: center; border-collapse: collapse;" id="table{i}"'
                    )
                    html_table = html_table.replace('<th', '<th style="border: 1px solid black; padding: 5px;"')
                    html_table = html_table.replace('<td', '<td style="border: 1px solid black; padding: 5px;"')
                else:  # If it's already an HTML string
                    html_table = table
                html_tables.append(html_table)

        # Combine body and tables
        full_body = body
        for i, html_table in enumerate(html_tables):
            full_body += f"<br><br>{html_table}"

        mail.HTMLBody = full_body

        if auto_send:
            mail.Send()
        else:
            mail.Display(True)

    except Exception as e:
        print(f"An error occurred: {str(e)}")
        print(f"Error type: {type(e).__name__}")
        print(f"Error details: {sys.exc_info()}")

import datetime
current_month = datetime.datetime.now().strftime("%B")

# Example usage
subject = "Machine Learning World Equity Portfolio for {current_month} 2025 and Monthly Performance Analysis"
recipients = ["recipient1@example.com", "recipient2@example.com"]
body = f"""
<p>Bonjour à tous,</p>
<p>   </p>
<p>Suivi de notre portefeuille Machine Learning Action World pour implémentation sur le mois {current_month} 2025.</p>
<p>Univers investissable : MSCI WORLD</p>
<p>   </p>
<p>   </p>
<p>   </p>
<h3 style="text-decoration: underline;">Methodology</h3>
<p>We score companies on different investment styles like Growth (5y past growth on Sales and EPS & FY1 Growth Income, EPS, Sales), Valuation (PE, PB, Price to CF, EV/Ebitda, EV/Sales : L12M and FY1), Balance Sheet Quality (ROE, ROTE, Operating Margin, Leverage, Earnings variability).</p>
<p>We used these scores and their dynamics (3M, 6M and 12M : this allows us to capture not just static financial positions but also the trajectory) as inputs in our machine learning models to find companies with high potential.</p>
<p>We use XGBoost, an advanced algorithm, to find relationship between past financial data and performances the month after.</p>
<p>You can find attached the Methodology</p>
<p>   </p>
<p>   </p>
<p>   </p>
<h3 style="text-decoration: underline;">Current Position of the TOP Portfolio for {current_month} 2025 :</h3>
<p>Here is the full list of Top basket, Excel version is also attached for your reference.</p>
<p>Key Metrics in the column and Best Analyst Target Price and %Analyst Upside Price</p>
<img src="cid:basket_top" style="width:70%; max-width:800px;">
<p>   </p>
<p>   </p>
<p>   </p>
<h3 style="text-decoration: underline;">Periodic Performance (%)</h3>
<img src="cid:periodic_perf" style="width:70%; max-width:800px;"><br>
<img src="cid:perf_plot" style="width:70%; max_width:800px;">
<p>Portfolio TOP and WORST are tickerized in Bloomberg, we can open the access on request to follow membership and performances.</p>
<p>   </p>
<h3 style="text-decoration: underline;">Metrics</h3>
<img src="cid:metrics" style="width:70%; max-width:800px;">
<p>   </p>
<h3 style="text-decoration: underline;">Top 25 and Worst 25 Performing Companies of Top</h3>
<img src="cid:top_25_weekly" style="width:70%; max-width:800px;">
<img src="cid:worst_25_weekly" style="width:70%; max-width:800px;">
<p>   </p>
<h3 style="text-decoration: underline;">Comparison per Sector</h3>
<img src="cid:weight_comparison_secto" style="width:70%; max-width:800px;"><p>   </p>
<img src="cid:rdt_comparison_secto" style="width:70%; max-width:800px;">
<p>   </p>
<p>Best regards,</p>
"""

attachments = [os.path.join(mail_output_path, r"basket_et_shap.xlsx")]
embedded_images = {
    "perf_plot": os.path.join(mail_output_path, r"perf_plot.png"),
    "periodic_perf": os.path.join(mail_output_path, r"periodic_perf.png"),
    "metrics": os.path.join(mail_output_path, r"metrics.png"),
    "top_25_weekly": os.path.join(mail_output_path, r"top_25_weekly.png"),
    "worst_25_weekly": os.path.join(mail_output_path, r"worst_25_weekly.png"),
    "weight_comparison_secto": os.path.join(mail_output_path, r"weight_comparison_secto.png"),
    "rdt_comparison_secto": os.path.join(mail_output_path, r"rdt_comparaison_secto.png"),
    # "basket_top": os.path.join(mail_output_path, r"basket_top.png")
}

generate_and_send_mail(subject, recipients, body, attachments, embedded_images)